# Step 13 · natural-text white-box attack (fluency-constrained)
Same white-box attack as Steps 6/11 (30% word budget, 100 steps, payload protected, agent no-poison check), but every
swap must be one of **distilroberta-base's top-50 predictions in context**, and the description's **GPT-2
log-perplexity may rise by at most 0.25 nats/token** (unconstrained evaders rose ~1.2).

Run this notebook **twice at the same time**: once with `BACKBONE = "deb"`, once with `BACKBONE = "e5"`
(Kaggle allows 2 concurrent sessions). Each copy (~4–6 h):
1. `adv_train.py` (seed 42): round-2 ADV checkpoint = the final system's detector (~12 min).
2. Natural attack vs the **base** detector, global threshold (does a fluent evasion exist at all?).
3. Natural attack vs the **final system** (round-2 ADV + calibrated per-server threshold).
4. `defense_checks.py` fluency part: GPT-2 filter flag rates + median NLL for natural vs unconstrained evaders.

**Settings:** GPU **T4 x2** · Internet **On** · Secrets **Hugging Face** + **mcp-git** ticked · Save & Run All.
After both finish: Output → Download All for each, put both zips in the Project folder.

In [ ]:
# ---- settings for this run ----
BACKBONE = "deb"           # "deb" or "e5"  <- change for the second copy
REPO = "github.com/Arn15/mcp-poison-detect"
BRANCH = "main"
BB = {"deb": "microsoft/deberta-v3-base", "e5": "intfloat/multilingual-e5-base"}[BACKBONE]
SRC_TRAIN = f"[results/wb1/whitebox.jsonl,results/wb1e5/whitebox.jsonl,results/wb3_base_b30_{BACKBONE}/whitebox.jsonl,results/wb3_adv_b30_{BACKBONE}/whitebox.jsonl]"
NAT = "whitebox.natural=true whitebox.max_frac=0.30 whitebox.steps=100 whitebox.mlm_k=50 whitebox.max_dnll=0.25 whitebox.max_hours=5"
UNCONSTRAINED = {"deb": "results/wb3_base_b30_deb", "e5": "results/wb3_base_b30_e5"}[BACKBONE]   # comparison run
KEEP_CKPT = False
GPU_DEPS = True
MIN_COMMIT_FILE = "notebooks/experiments/kaggle_step13_natural.ipynb"

In [ ]:
import os, subprocess, base64
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
def secret(k):
    try: return sec.get_secret(k)
    except Exception: return None
HF_SECRET, GH_SECRET = "Hugging Face", "mcp-git"   # names of your Kaggle secrets
os.environ["HF_TOKEN"] = secret(HF_SECRET) or ""
gh = secret(GH_SECRET)
assert gh, "Secret 'mcp-git' not attached to this notebook (Add-ons -> Secrets)"
WORK = "/kaggle/working"

# Token goes in an HTTP header for this command only: never in the URL, .git/config, or error output.
b64 = base64.b64encode(f"x-access-token:{gh}".encode()).decode()
auth = ["-c", f"http.extraHeader=Authorization: Basic {b64}"]
def git(*args, cwd=None):
    r = subprocess.run(["git", *auth, *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(gh, "***")
        raise RuntimeError(f"git {args[0]} failed: {msg.strip()}")
    return r.stdout

os.chdir(WORK)
repo_dir = os.path.join(WORK, "mcp-poison-detect")
if os.path.isdir(os.path.join(repo_dir, ".git")):
    git("fetch", "-q", cwd=repo_dir); git("checkout", "-q", BRANCH, cwd=repo_dir); git("pull", "-q", cwd=repo_dir)
else:
    git("clone", "-q", "-b", BRANCH, f"https://{REPO}.git", repo_dir)
os.chdir(repo_dir)
print("commit:", git("log", "-1", "--oneline", cwd=repo_dir))
assert os.path.exists(MIN_COMMIT_FILE), f"{MIN_COMMIT_FILE} missing: push the latest commit first (needs the natural-mode attacker)"

In [ ]:
!pip install -q -r requirements.txt
if GPU_DEPS:
    !pip install -q -r requirements-gpu.txt
!pip install -q -e . --no-deps

In [ ]:
import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv
assert torch.cuda.device_count() >= 2, "Need GPU T4 x2"
# 1. final-system detector (round-2 ADV, seed 42)
!python scripts/adv_train.py --tag adv2_{BACKBONE}_ckpt --set detector.backbone={BB} 'advtrain.sources={SRC_TRAIN}' 'advtrain.seeds=[42]'

In [ ]:
# 2. natural attack vs the BASE detector (trained in-run, global threshold)
!python scripts/attack_whitebox.py --tag nat_base_{BACKBONE} --set detector.backbone={BB} {NAT}

In [ ]:
# 3. natural attack vs the FINAL system (round-2 ADV + calibrated per-server threshold)
import glob, os
adv_run = sorted(glob.glob(f"/kaggle/working/outputs/adv_train/*adv2_{BACKBONE}_ckpt"))[-1]
ckpt = os.path.join(adv_run, "ckpt_adv")
assert os.path.exists(os.path.join(ckpt, "detector.json")), ckpt
!python scripts/attack_whitebox.py --tag nat_final_cal_{BACKBONE} --set detector.backbone={BB} whitebox.detector_ckpt={ckpt} whitebox.calibrated=true {NAT}

In [ ]:
# 4. fluency filter: natural vs unconstrained evaders
O = "/kaggle/working/outputs/attack_whitebox"
nat_runs = [sorted(glob.glob(f"{O}/*nat_base_{BACKBONE}"))[-1], sorted(glob.glob(f"{O}/*nat_final_cal_{BACKBONE}"))[-1]]
RUNS = "[" + ",".join([UNCONSTRAINED] + nat_runs) + "]"
!python scripts/defense_checks.py --tag fluency_nat_{BACKBONE} --set 'defense.parts=[fluency]' 'defense.wb_runs={RUNS}'

In [ ]:
# summary
import json, shutil
for r in nat_runs:
    s = json.load(open(f"{r}/summary.json"))
    print(os.path.basename(r), "| calibrated", s["calibrated"], "| evaded", s["n_evaded"], "of", s["n_items"],
          "| genuine", s.get("n_genuine_evasions"), "| words changed", s["mean_words_changed_evaders"],
          "| median dNLL", s.get("median_dnll_evaders"))
    rows = [json.loads(l) for l in open(f"{r}/whitebox.jsonl")]
    for x in [x for x in rows if x["evaded"]][:3]:
        print("   ", x["changes"])
fr = sorted(glob.glob("/kaggle/working/outputs/defense_checks/*"))[-1]
print(open(f"{fr}/defense_results.md").read())
if not KEEP_CKPT:
    for d in glob.glob("/kaggle/working/outputs/**/ckpt_*", recursive=True):
        shutil.rmtree(d, ignore_errors=True)